# Finding Insights from Data with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fathanick/Fundamentals-of-Data-Science/blob/main/Data%20Visualization/06_Finding_Insights_from_Data_Tutorial.ipynb)

---

**Course**: Fundamentals of Data Science  
**Instructor**: Ahmad Fathan Hidayatullah, S.T., M.Cs., Ph.D.  
**Institution**: Universitas Islam Indonesia (UII)

---

An **insight** is a meaningful understanding or actionable conclusion gained from analyzing data. It answers:
- *What is happening?*
- *Why is it happening?*
- *What should we do next?*

In this tutorial, we follow the **Data → Analytics → Insights → Action** pipeline to extract real, actionable insights from a supermarket sales dataset.

**What you will learn:**
1. The difference between data, information, and insight
2. How to ask the right analytical questions
3. How to combine visualizations with written interpretation
4. How to present findings as actionable recommendations

---
## Data vs Information vs Insight

These three concepts are often confused. Here is how they differ:

| Level | Example |
|---|---|
| **Data** | "Branch A sold 340 units this month." |
| **Information** | "Branch A's sales dropped 15% compared to last month." |
| **Insight** | "Branch A's sales dropped because a competitor opened nearby → we should launch a loyalty program." |

Raw data becomes **information** when we add context (comparison, trend).  
Information becomes an **insight** when we understand the *why* and connect it to a *decision*.

> **Goal**: In this tutorial, we move all the way from data to insights.

In [ ]:
# Install libraries (if needed)
# !pip install matplotlib seaborn pandas numpy

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

---
## Load the Dataset

We use the same **Supermarket Sales** dataset (1,000 transactions). We also engineer a few new columns for time-based analysis.

In [ ]:
url = "https://raw.githubusercontent.com/fathanick/Fundamentals-of-Data-Science/main/Data%20Visualization/supermarket_sales.csv"
df = pd.read_csv(url)
df['Date'] = pd.to_datetime(df['Date'])
df['Hour'] = df['Time'].apply(lambda x: int(x.split(':')[0]))
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()
df.head()

---
## Step 1 — Explore the Data

Before looking for insights, we must **understand the data**:
- What columns do we have?
- Are there missing values?
- What are the basic statistics?

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.describe()

---
## Step 2 — Ask the Right Questions

Good insights come from asking the **right questions**. Here are five analytical questions we will explore:

1. Which branch performs best in terms of revenue and customer satisfaction?
2. What product lines generate the most revenue?
3. Is there a relationship between customer type and spending?
4. When do most customers shop? (time of day, day of week)
5. Do payment methods differ across branches?

> **Tip**: Always start with a business question, not the data. The question guides what to look for.

---
## Insight 1: Branch Performance

**Question**: *Which branch performs best in terms of revenue and customer satisfaction?*

Let's compare the three branches (A, B, C) across revenue, rating, and number of transactions.

In [ ]:
branch_summary = df.groupby('Branch').agg(
    total_revenue=('Total', 'sum'),
    avg_rating=('Rating', 'mean'),
    num_transactions=('Invoice ID', 'count')
).round(2)

print(branch_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Revenue
branch_summary['total_revenue'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Revenue by Branch')
axes[0].set_ylabel('Revenue (USD)')

# Rating
branch_summary['avg_rating'].plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Average Rating by Branch')
axes[1].set_ylabel('Rating')
axes[1].set_ylim(6, 8)

# Transactions
branch_summary['num_transactions'].plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Number of Transactions by Branch')
axes[2].set_ylabel('Count')

for ax in axes:
    ax.set_xlabel('Branch')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

> **Insight**: All three branches have similar revenue and transaction counts, indicating **balanced store performance** across locations. However, Branch C has a slightly higher average rating, suggesting better customer experience at that location.
>
> **Possible Action**: Investigate what Branch C does differently in terms of service quality, staff training, or store layout — and replicate those practices across Branches A and B.

---
## Insight 2: Product Line Analysis

**Question**: *What product lines generate the most revenue — and do high-revenue lines also earn high ratings?*

In [ ]:
product_summary = df.groupby('Product line').agg(
    total_revenue=('Total', 'sum'),
    avg_unit_price=('Unit price', 'mean'),
    total_quantity=('Quantity', 'sum'),
    avg_rating=('Rating', 'mean')
).round(2).sort_values('total_revenue', ascending=False)

print(product_summary)

In [ ]:
plt.figure(figsize=(8, 5))
product_summary['total_revenue'].sort_values().plot(kind='barh', color='teal')
plt.xlabel('Total Revenue (USD)')
plt.title('Revenue by Product Line')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=product_summary.reset_index(), x='avg_rating', y='total_revenue',
                size='total_quantity', sizes=(100, 500), hue='Product line', legend='brief')
plt.xlabel('Average Rating')
plt.ylabel('Total Revenue (USD)')
plt.title('Rating vs Revenue by Product Line')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

> **Insight**: The top-revenue product lines do not always match the highest-rated lines — **high revenue does not always mean high satisfaction**. Some product lines earn well but receive mediocre ratings, suggesting customers may feel the value-for-money is not optimal.
>
> **Possible Action**:
> - For **high-rating but low-revenue** lines: run promotional campaigns to increase awareness and sales volume.
> - For **high-revenue but lower-rating** lines: investigate product quality, pricing fairness, or post-purchase service issues.

---
## Insight 3: When Do Customers Shop?

**Question**: *Are there peak hours or days? How can the store use this information?*

In [ ]:
hourly = df.groupby('Hour')['Invoice ID'].count()

plt.figure(figsize=(10, 5))
hourly.plot(kind='bar', color='coral')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Transactions')
plt.title('Transaction Volume by Hour of Day')
plt.tight_layout()
plt.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = df.pivot_table(values='Total', index='DayOfWeek', columns='Hour', aggfunc='count')
heatmap_data = heatmap_data.reindex(day_order)

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='Blues', linewidths=0.5)
plt.title('Number of Transactions by Day and Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

> **Insight**: Peak shopping hours are around **1 PM (13:00)** and **7 PM (19:00)**. Saturday afternoons and Tuesday evenings show the highest transaction density in the heatmap.
>
> **Possible Action**:
> - Schedule **more staff** during identified peak hours to maintain service quality.
> - Run **time-limited promotions** (flash sales, discount codes) during off-peak hours to spread traffic more evenly.

---
## Insight 4: Customer Type and Spending

**Question**: *Do members spend more than non-members? Is the membership program effective?*

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Customer type', y='Total', palette='Set2')
plt.title('Spending Distribution by Customer Type')
plt.ylabel('Total Spending (USD)')
plt.tight_layout()
plt.show()

In [ ]:
ct_summary = df.groupby(['Customer type', 'Gender']).agg(
    avg_spending=('Total', 'mean'),
    count=('Invoice ID', 'count')
).round(2)
print(ct_summary)

> **Insight**: Members and non-members spend roughly the **same amount on average**. The current membership program does not appear to be driving higher spending behavior — members are not purchasing more than regular customers.
>
> **Possible Action**: Redesign the membership program to include more attractive perks — such as exclusive discounts, point rewards, or early access to promotions — specifically designed to incentivize higher spending from members.

---
## Insight 5: Payment Methods Across Branches

**Question**: *Do customers at different branches prefer different payment methods?*

In [ ]:
payment_branch = pd.crosstab(df['Branch'], df['Payment'], normalize='index') * 100

payment_branch.plot(kind='bar', stacked=True, figsize=(8, 5), colormap='Set2')
plt.title('Payment Method Distribution by Branch (%)')
plt.ylabel('Percentage')
plt.xlabel('Branch')
plt.legend(title='Payment', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> **Insight**: Payment preferences are relatively balanced across all branches, but Branch A shows a noticeably **higher proportion of e-wallet usage** compared to B and C, where cash and credit card are more prevalent.
>
> **Possible Action**: Partner with popular e-wallet providers (e.g., GoPay, OVO, DANA) for **exclusive cashback or loyalty promotions** at branches where e-wallet adoption is already high — capitalizing on existing customer behavior.

---
## Summarizing Insights

A good analyst does not just find insights — they **communicate them clearly**. Here is a summary of all five insights:

| # | Question | Key Finding | Recommended Action |
|---|---|---|---|
| 1 | Which branch performs best? | All branches are balanced; Branch C has slightly higher ratings | Replicate Branch C's service practices |
| 2 | What product lines generate most revenue? | Revenue and rating rankings differ across product lines | Promote high-rating lines; investigate quality in high-revenue/low-rating lines |
| 3 | When do customers shop most? | Peaks at 1 PM and 7 PM; Saturday afternoons busiest | Add staff at peak hours; run off-peak promotions |
| 4 | Do members spend more? | No significant difference vs non-members | Redesign membership benefits to drive higher spending |
| 5 | Payment preferences by branch? | Branch A uses more e-wallets | Partner with e-wallet providers for targeted promotions |

> **Key lesson**: The skill is not just creating charts — it is **interpreting what they mean** and recommending concrete actions.

---
## Case Study: Analyzing Online Course Reviews

You work for an **online learning platform**. Management wants to know:
- Which courses perform well?
- What topics get low ratings?
- When do students enroll the most?

Your task is to find insights and present them as actionable recommendations to the CEO.

### Step 1: Create the Dataset

In [ ]:
np.random.seed(123)
n = 300

courses = ['Python Basics', 'Data Science', 'Web Development', 'Machine Learning', 'Database']
categories = ['Programming', 'Data', 'Web', 'Data', 'Database']
course_cat = dict(zip(courses, categories))

reviews = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=180).repeat(n//180).reset_index(drop=True)[:n],
    'course': np.random.choice(courses, n),
    'rating': np.round(np.clip(np.random.normal(3.8, 0.8, n), 1, 5), 1),
    'completion_rate': np.round(np.clip(np.random.normal(0.65, 0.2, n), 0, 1), 2),
    'price': np.random.choice([99000, 149000, 199000, 249000, 299000], n)
})
reviews['category'] = reviews['course'].map(course_cat)
reviews['month'] = reviews['date'].dt.month_name()
reviews['revenue'] = reviews['price'] * reviews['completion_rate']
reviews.head()

### Task 1: Explore

How many reviews does each course have? What is the average rating and average completion rate per course?

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
course_summary = reviews.groupby('course').agg(
    num_reviews=('rating', 'count'),
    avg_rating=('rating', 'mean'),
    avg_completion=('completion_rate', 'mean'),
    total_revenue=('revenue', 'sum')
).round(2).sort_values('avg_rating', ascending=False)

print(course_summary)

### Task 2: Visualize

a) Create a **bar chart** of average rating per course (sorted from highest to lowest).  
b) Create a **line chart** showing the number of enrollments per month.

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION — a) Bar chart of average ratings
course_ratings = reviews.groupby('course')['rating'].mean().sort_values(ascending=True)

plt.figure(figsize=(8, 5))
course_ratings.plot(kind='barh', color='mediumslateblue')
plt.xlabel('Average Rating')
plt.title('Average Rating by Course')
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION — b) Monthly enrollment line chart
month_order = ['January', 'February', 'March', 'April', 'May', 'June']
monthly_enrollments = reviews.groupby('month')['course'].count().reindex(month_order)

plt.figure(figsize=(10, 5))
plt.plot(monthly_enrollments.index, monthly_enrollments.values, marker='o', linewidth=2, color='darkorange')
plt.xlabel('Month')
plt.ylabel('Number of Enrollments')
plt.title('Monthly Course Enrollments (Jan–Jun 2024)')
plt.tight_layout()
plt.show()

### Task 3: Find Insights

Look at your exploration and visualizations. Then answer:
- Which course has the **highest rating** but the **lowest enrollment**?
- What action would you recommend to the platform based on this finding?

Write your code to identify the answer, then write your recommendation in the markdown cell below.

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
course_comparison = reviews.groupby('course').agg(
    avg_rating=('rating', 'mean'),
    enrollment=('rating', 'count')
).round(2)

highest_rated = course_comparison['avg_rating'].idxmax()
lowest_enrolled = course_comparison['enrollment'].idxmin()

print(f"Highest rated course  : {highest_rated} (rating: {course_comparison.loc[highest_rated, 'avg_rating']})")
print(f"Lowest enrolled course: {lowest_enrolled} (enrollments: {course_comparison.loc[lowest_enrolled, 'enrollment']})")
print()
print(course_comparison.sort_values('avg_rating', ascending=False))

**Your Insight and Recommendation** *(write here)*:

> ...

### Task 4: Communicate Your Findings

Write a **3-sentence summary** of your findings as if you were reporting to the platform's CEO. Your summary should:
1. State the most important finding.
2. Explain what it means for the business.
3. Recommend a concrete next step.

*Write your response in the cell below.*

**Executive Summary** *(write here)*:

> ...

**Sample Answer**:

> Our analysis of 300 course enrollments from January to June 2024 reveals that the *Database* course consistently earns the highest ratings but has the lowest enrollment count — indicating strong quality that is not reaching its potential audience. Despite delivering a superior learner experience, this course is underperforming commercially, likely due to limited discoverability or marketing. We recommend launching a targeted promotional campaign for the Database course, including featuring it on the platform homepage and offering a limited-time introductory discount to attract new learners.

---
## What Makes a Good Insight? — Closing Checklist

Before presenting any insight, ask yourself:

- ✅ It is **based on data**, not just intuition.
- ✅ It **answers a specific question** that matters to the business.
- ✅ It is **actionable** — someone can make a decision based on it.
- ✅ It is **clearly communicated** using visuals + written interpretation.

---

### The Data → Analytics → Insights → Action Pipeline

```
DATA           →   ANALYTICS      →   INSIGHTS          →   ACTION
(raw numbers)      (explore,          (what it means        (what to do)
                    visualize)         for the business)
```

> *"Data is the new oil — but raw oil is useless. You need to refine it into insights before it powers anything."*

Congratulations on completing the **Finding Insights from Data** tutorial!